# Notebook 3: Klasyczne metody uczenia maszynowego (ML)

## Cel
Implementacja i ocena 5 klasycznych algorytmów ML do klasyfikacji EKG:
1. **Regresja logistyczna** (Logistic Regression)
2. **Las losowy** (Random Forest)
3. **Maszyna wektorów nośnych** (SVM z jądrem RBF)
4. **k-Nearest Neighbors** (KNN, k=5)
5. **Gradient Boosting**
6. **Naiwny Bayes** (model bazowy – punkt odniesienia)

Dane wejściowe: wektory 228 cech statystycznych (z Notebook 2).
Ocena bez strojenia hiperparametrów (Kontrola 2).

**Wymaganie:** Kontrola pośrednia nr 2 – wszystkie algorytmy (bez optymalizacji)


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, f1_score, ConfusionMatrixDisplay,
)

sys.path.insert(0, '..')
from src.data_loader import (
    load_ptbxl_metadata, build_labels, load_raw_data,
    get_train_val_test_split, SUPERCLASSES,
)
from src.preprocessing import preprocess_pipeline, preprocess_batch
from src.feature_extraction import extract_features_batch, get_feature_names
from src.classical_models import get_classical_models, evaluate_all_models, train_evaluate_model

plt.rcParams['figure.dpi'] = 100
sns.set_theme(style='whitegrid')

DATA_PATH = '../data/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3'
FS = 100

print("Moduły załadowane.")


## 1. Wczytanie i przygotowanie danych

Wczytujemy dane, przetwarzamy je potok przetwarzania i ekstrahujemy cechy.
Używamy podzbioru 500 próbek (100 z każdej klasy) do szybkiej demonstracji.
Dla pełnego eksperymentu można usunąć limit `n_per_class`.


In [ ]:
# Wczytaj metadane
Y = load_ptbxl_metadata(DATA_PATH)
Y = build_labels(Y, DATA_PATH)
train_idx, val_idx, test_idx = get_train_val_test_split(Y)

# ── Podzbiór do demonstracji (100 próbek na klasę) ────────────────────────
# Zmień n_per_class=None, żeby użyć PEŁNEGO zbioru (zajmie kilka minut)
N_PER_CLASS = 100

def sample_per_class(df, idx, n=None):
    sampled = []
    for cls in SUPERCLASSES:
        cls_idx = df[(df.label_single == cls) & df.index.isin(idx)].index
        sampled.extend(cls_idx[:n] if n else cls_idx)
    return df.loc[sampled]

Y_train_s = sample_per_class(Y, train_idx, N_PER_CLASS)
Y_test_s  = sample_per_class(Y, test_idx,  N_PER_CLASS // 5)

print(f"Podzbiór treningowy: {len(Y_train_s):,} próbek")
print(f"Podzbiór testowy:    {len(Y_test_s):,} próbek")


In [ ]:
# Wczytaj surowe sygnały i przetwórz
print("Ładowanie sygnałów...")
X_train_raw = load_raw_data(Y_train_s, sampling_rate=FS, data_path=DATA_PATH)
X_test_raw  = load_raw_data(Y_test_s,  sampling_rate=FS, data_path=DATA_PATH)

print("Przetwarzanie potoku...")
X_train_proc = preprocess_batch(X_train_raw, fs=FS, verbose=False)
X_test_proc  = preprocess_batch(X_test_raw,  fs=FS, verbose=False)

print("Ekstrakcja cech...")
X_train = extract_features_batch(X_train_proc, fs=FS, verbose=False)
X_test  = extract_features_batch(X_test_proc,  fs=FS, verbose=False)

# Napraw NaN/Inf
X_train = np.nan_to_num(X_train, nan=0.0, posinf=0.0, neginf=0.0)
X_test  = np.nan_to_num(X_test,  nan=0.0, posinf=0.0, neginf=0.0)

# Koduj etykiety jako liczby całkowite
le = LabelEncoder()
le.fit(SUPERCLASSES)
y_train = le.transform(Y_train_s['label_single'].values)
y_test  = le.transform(Y_test_s['label_single'].values)

print(f"\nX_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_test:  {X_test.shape},  y_test:  {y_test.shape}")
print(f"Klasy: {le.classes_}")


## 2. Trening i ewaluacja modeli

In [ ]:
# Uruchom wszystkie modele i zbierz wyniki
summary_df, all_results = evaluate_all_models(
    X_train, y_train, X_test, y_test,
    classes=list(le.classes_)
)


## 3. Porównanie wyników

In [ ]:
# Tabela porównawcza
print("\n=== TABELA PORÓWNAWCZA MODELI ===")
display_cols = ['Model', 'Accuracy', 'Balanced Accuracy', 'F1 Macro', 'F1 Weighted', 'Train Time [s]']
print(summary_df[display_cols].to_string(index=False))


In [ ]:
# Wykres słupkowy metryk
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
models = summary_df['Model'].tolist()
x = np.arange(len(models))
width = 0.35

# F1-score
axes[0].bar(x - width/2, summary_df['F1 Macro'],    width, label='F1 Macro',    color='steelblue', alpha=0.85)
axes[0].bar(x + width/2, summary_df['F1 Weighted'], width, label='F1 Weighted', color='coral',     alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels(models, rotation=25, ha='right')
axes[0].set_ylabel('F1-score')
axes[0].set_title('Porównanie F1-score modeli', fontweight='bold')
axes[0].legend()
axes[0].set_ylim(0, 1)

# Accuracy vs. Balanced Accuracy
axes[1].bar(x - width/2, summary_df['Accuracy'],          width, label='Accuracy',          color='forestgreen', alpha=0.85)
axes[1].bar(x + width/2, summary_df['Balanced Accuracy'], width, label='Balanced Accuracy', color='orange',      alpha=0.85)
axes[1].set_xticks(x)
axes[1].set_xticklabels(models, rotation=25, ha='right')
axes[1].set_ylabel('Dokładność')
axes[1].set_title('Accuracy vs Balanced Accuracy', fontweight='bold')
axes[1].legend()
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.savefig('../results/classical_ml_comparison.png', bbox_inches='tight', dpi=150)
plt.show()


## 4. Macierze pomyłek (Confusion Matrices)

In [ ]:
# Macierze pomyłek dla wszystkich modeli
n_models = len(all_results)
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

for ax, (name, result) in zip(axes.flatten(), all_results.items()):
    cm = result['confusion_matrix']
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
    disp.plot(ax=ax, colorbar=False, cmap='Blues', values_format='d')
    ax.set_title(f'{name}\nAcc={result["accuracy"]:.3f} | F1={result["f1_macro"]:.3f}',
                 fontweight='bold', fontsize=10)

# Ukryj ostatni pusty wykres jeśli modeli jest mniej niż 6
for i in range(len(all_results), 6):
    axes.flatten()[i].set_visible(False)

plt.tight_layout()
plt.savefig('../results/classical_ml_confusion_matrices.png', bbox_inches='tight', dpi=150)
plt.show()


## 5. Analiza ważności cech (Random Forest)

Las losowy dostarcza miarę ważności cech (feature importance) – które cechy
statystyczne są najbardziej pomocne w klasyfikacji.


In [ ]:
# Feature importance z Random Forest
rf_model = all_results['Random Forest']['model']
importances = rf_model.feature_importances_
feature_names = get_feature_names()

# Top 20 najważniejszych cech
top_n = 20
top_idx = np.argsort(importances)[::-1][:top_n]

fig, ax = plt.subplots(figsize=(12, 7))
ax.barh(
    [feature_names[i] for i in top_idx[::-1]],
    importances[top_idx[::-1]],
    color='steelblue', alpha=0.85
)
ax.set_xlabel('Ważność cechy (Gini impurity)')
ax.set_title(f'Top {top_n} najważniejszych cech – Random Forest', fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('../results/classical_ml_feature_importance.png', bbox_inches='tight', dpi=150)
plt.show()

print("\nTop 10 najważniejszych cech:")
for rank, i in enumerate(top_idx[:10], 1):
    print(f"  {rank:2d}. {feature_names[i]:35s} = {importances[i]:.4f}")


In [ ]:
# Zapis najlepszego modelu
import pickle
best_model_name = summary_df.iloc[0]['Model']
best_model = all_results[best_model_name]['model']

os.makedirs('../models', exist_ok=True)
with open('../models/best_classical_model.pkl', 'wb') as f:
    pickle.dump({'model': best_model, 'model_name': best_model_name, 'label_encoder': le}, f)

print(f"Najlepszy model: {best_model_name}")
print(f"  F1 Macro: {summary_df.iloc[0]['F1 Macro']:.4f}")
print(f"  Accuracy: {summary_df.iloc[0]['Accuracy']:.4f}")
print(f"Zapisano do: models/best_classical_model.pkl")


---
## 6. Dostrajanie hiperparametrów (GridSearchCV)

Dla każdego modelu przeszukujemy siatkę parametrów używając **3-foldowej walidacji krzyżowej**.
Optymalizowana metryka: **F1-macro** (ważna przy niezbalansowanych klasach).

Wizualizacje:
- **Heatmapy** – jak kombinacja dwóch parametrów wpływa na F1-score (np. C × gamma dla SVM)
- **Wykresy liniowe** – wpływ jednego parametru (np. C w regresji logistycznej)
- **Porównanie przed/po** – wyniki domyślnych vs optymalnych parametrów

In [ ]:
from src.classical_models import tune_hyperparameters

# ── Uruchom GridSearchCV dla wszystkich modeli ──────────────────────────────
# cv=3 (3-foldowa walidacja krzyżowa) – kompromis dokładność/czas
# Dla pełnego zbioru danych może to zająć kilka minut
print("Rozpoczynam dostrajanie hiperparametrów...")
print("(cv=3, scoring='f1_macro', n_jobs=-1)\n")

tuning_results = tune_hyperparameters(X_train, y_train, cv=3, verbose=True)

print("\n✓ GridSearchCV zakończony dla wszystkich modeli.")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

# ═══════════════════════════════════════════════════════════════════════════
# WIZUALIZACJA 1: Heatmapa SVM (C × gamma)
# ═══════════════════════════════════════════════════════════════════════════
# SVM ma dwa kluczowe parametry:
#   C     – regularyzacja: duże C = mniejszy margines, bardziej dopasowany
#   gamma – zasięg jądra RBF: małe gamma = szerszy wpływ punktów
# Heatmapa pokazuje jak każda kombinacja (C, gamma) wpływa na F1-macro (CV)

svm_gs = tuning_results['SVM (RBF)']['grid_search']
cv_res = svm_gs.cv_results_

# Wyciągnij wartości parametrów i wyniki
C_vals     = [0.1, 1.0, 10.0]
gamma_vals = ['scale', 0.001, 0.01]
scores_matrix = np.array(cv_res['mean_test_score']).reshape(len(C_vals), len(gamma_vals))

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(scores_matrix, cmap='YlOrRd', aspect='auto', vmin=scores_matrix.min() - 0.02)
plt.colorbar(im, ax=ax, label='F1-macro (CV)')

ax.set_xticks(range(len(gamma_vals)))
ax.set_yticks(range(len(C_vals)))
ax.set_xticklabels(gamma_vals, fontsize=11)
ax.set_yticklabels(C_vals, fontsize=11)
ax.set_xlabel('gamma', fontsize=12)
ax.set_ylabel('C', fontsize=12)
ax.set_title('SVM (RBF): GridSearchCV – F1-macro\n(heatmapa C × gamma)', fontweight='bold')

# Annotate każdą komórkę wartością
for i in range(len(C_vals)):
    for j in range(len(gamma_vals)):
        ax.text(j, i, f'{scores_matrix[i, j]:.3f}',
                ha='center', va='center', fontsize=11,
                color='black' if scores_matrix[i, j] < scores_matrix.max() - 0.05 else 'white',
                fontweight='bold')

best_params = tuning_results['SVM (RBF)']['best_params']
ax.set_title(
    f"SVM (RBF): GridSearchCV – F1-macro\n"
    f"Najlepsze: C={best_params.get('clf__C')}, gamma={best_params.get('clf__gamma')} "
    f"→ F1={tuning_results['SVM (RBF)']['best_score']:.4f}",
    fontweight='bold'
)

plt.tight_layout()
plt.savefig('../results/tuning_svm_heatmap.png', bbox_inches='tight', dpi=150)
plt.show()
print(f"Najlepsze parametry SVM: {best_params}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# WIZUALIZACJA 2: Heatmapa Random Forest (n_estimators × max_depth)
# ═══════════════════════════════════════════════════════════════════════════
# RF ma dwa kluczowe parametry:
#   n_estimators – liczba drzew: więcej = stabilniej, ale wolniej
#   max_depth    – głębokość drzewa: None = pełne drzewa (ryzyko overfitting)

rf_gs = tuning_results['Random Forest']['grid_search']
cv_res_rf = rf_gs.cv_results_

n_est_vals  = [50, 100, 200]
depth_vals  = [None, 10, 20]
scores_rf   = np.array(cv_res_rf['mean_test_score']).reshape(len(n_est_vals), len(depth_vals))

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(scores_rf, cmap='Blues', aspect='auto', vmin=scores_rf.min() - 0.02)
plt.colorbar(im, ax=ax, label='F1-macro (CV)')

ax.set_xticks(range(len(depth_vals)))
ax.set_yticks(range(len(n_est_vals)))
ax.set_xticklabels(['brak limitu', '10', '20'], fontsize=11)
ax.set_yticklabels(n_est_vals, fontsize=11)
ax.set_xlabel('max_depth', fontsize=12)
ax.set_ylabel('n_estimators', fontsize=12)

for i in range(len(n_est_vals)):
    for j in range(len(depth_vals)):
        ax.text(j, i, f'{scores_rf[i, j]:.3f}',
                ha='center', va='center', fontsize=11, fontweight='bold',
                color='white' if scores_rf[i, j] > scores_rf.max() - 0.05 else 'black')

best_rf = tuning_results['Random Forest']['best_params']
ax.set_title(
    f"Random Forest: GridSearchCV – F1-macro\n"
    f"Najlepsze: n_estimators={best_rf.get('n_estimators')}, "
    f"max_depth={best_rf.get('max_depth')} "
    f"→ F1={tuning_results['Random Forest']['best_score']:.4f}",
    fontweight='bold'
)

plt.tight_layout()
plt.savefig('../results/tuning_rf_heatmap.png', bbox_inches='tight', dpi=150)
plt.show()
print(f"Najlepsze parametry RF: {best_rf}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# WIZUALIZACJA 3: Wykres liniowy – Regresja Logistyczna (C) i KNN (k)
# ═══════════════════════════════════════════════════════════════════════════
# Dla modeli z jednym kluczowym parametrem wykres liniowy jest czytelniejszy:
#   - LR: C kontroluje regularyzację; optymalne C widać jako "łokieć" krzywej
#   - KNN: k kontroluje lokalność; zbyt małe k → szum, zbyt duże → underfitting

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Regresja Logistyczna: C ──────────────────────────────────────────────
lr_gs   = tuning_results['Logistic Regression']['grid_search']
C_vals_lr  = [0.01, 0.1, 1.0, 10.0, 100.0]
lr_scores  = lr_gs.cv_results_['mean_test_score']
lr_std     = lr_gs.cv_results_['std_test_score']

axes[0].plot(C_vals_lr, lr_scores, 'o-', color='steelblue', linewidth=2, markersize=8, label='F1-macro (CV)')
axes[0].fill_between(C_vals_lr,
                     lr_scores - lr_std,
                     lr_scores + lr_std,
                     alpha=0.2, color='steelblue', label='±1 std')
best_C = tuning_results['Logistic Regression']['best_params'].get('clf__C')
best_lr_score = tuning_results['Logistic Regression']['best_score']
axes[0].axvline(best_C, color='red', linestyle='--', label=f'Najlepsze C={best_C}')
axes[0].set_xscale('log')
axes[0].set_xlabel('C (skala logarytmiczna)', fontsize=12)
axes[0].set_ylabel('F1-macro', fontsize=12)
axes[0].set_title(f'Regresja Logistyczna: wpływ C\nNajlepsze C={best_C} → F1={best_lr_score:.4f}', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# ── KNN: liczba sąsiadów k ──────────────────────────────────────────────
knn_gs  = tuning_results['KNN (k=5)']['grid_search']
k_vals  = [3, 5, 7, 11, 15]
metrics = ['euclidean', 'manhattan']
colors  = ['steelblue', 'coral']

for metric, color in zip(metrics, colors):
    # Filtruj wyniki dla tej metryki odległości
    mask = knn_gs.cv_results_['param_clf__metric'] == metric
    scores_k = knn_gs.cv_results_['mean_test_score'][mask]
    std_k    = knn_gs.cv_results_['std_test_score'][mask]
    axes[1].plot(k_vals, scores_k, 'o-', color=color, linewidth=2,
                 markersize=8, label=f'metric={metric}')
    axes[1].fill_between(k_vals, scores_k - std_k, scores_k + std_k,
                         alpha=0.15, color=color)

best_knn = tuning_results['KNN (k=5)']['best_params']
best_knn_score = tuning_results['KNN (k=5)']['best_score']
axes[1].axvline(best_knn.get('clf__n_neighbors'), color='red', linestyle='--',
                label=f"Najlepsze k={best_knn.get('clf__n_neighbors')}")
axes[1].set_xlabel('k (liczba sąsiadów)', fontsize=12)
axes[1].set_ylabel('F1-macro', fontsize=12)
axes[1].set_title(
    f"KNN: wpływ k i metryki odległości\n"
    f"Najlepsze k={best_knn.get('clf__n_neighbors')}, "
    f"metric={best_knn.get('clf__metric')} → F1={best_knn_score:.4f}",
    fontweight='bold'
)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/tuning_lr_knn_curves.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# WIZUALIZACJA 4: Heatmapa Gradient Boosting (learning_rate × max_depth)
# ═══════════════════════════════════════════════════════════════════════════
# Gradient Boosting jest wrażliwy na kombinację:
#   learning_rate – krok gradientu: małe lr + więcej drzew = lepszy model
#   max_depth     – głębokość drzew: zbyt głębokie drzewa → overfitting

gb_gs   = tuning_results['Gradient Boosting']['grid_search']
lr_vals  = [0.05, 0.1, 0.2]
d_vals   = [2, 3, 5]
scores_gb = np.array(gb_gs.cv_results_['mean_test_score']).reshape(len(lr_vals), len(d_vals))

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(scores_gb, cmap='Greens', aspect='auto', vmin=scores_gb.min() - 0.02)
plt.colorbar(im, ax=ax, label='F1-macro (CV)')

ax.set_xticks(range(len(d_vals)))
ax.set_yticks(range(len(lr_vals)))
ax.set_xticklabels(d_vals, fontsize=11)
ax.set_yticklabels(lr_vals, fontsize=11)
ax.set_xlabel('max_depth', fontsize=12)
ax.set_ylabel('learning_rate', fontsize=12)

for i in range(len(lr_vals)):
    for j in range(len(d_vals)):
        ax.text(j, i, f'{scores_gb[i, j]:.3f}',
                ha='center', va='center', fontsize=11, fontweight='bold',
                color='white' if scores_gb[i, j] > scores_gb.max() - 0.04 else 'black')

best_gb = tuning_results['Gradient Boosting']['best_params']
ax.set_title(
    f"Gradient Boosting: GridSearchCV – F1-macro\n"
    f"Najlepsze: lr={best_gb.get('learning_rate')}, "
    f"depth={best_gb.get('max_depth')} "
    f"→ F1={tuning_results['Gradient Boosting']['best_score']:.4f}",
    fontweight='bold'
)

plt.tight_layout()
plt.savefig('../results/tuning_gb_heatmap.png', bbox_inches='tight', dpi=150)
plt.show()
print(f"Najlepsze parametry GB: {best_gb}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# WIZUALIZACJA 5: Porównanie domyślne vs optymalne parametry (PRZED / PO)
# ═══════════════════════════════════════════════════════════════════════════
# Oceniamy każdy model na zbiorze testowym z:
#   (a) domyślnymi parametrami (z sekcji 2)
#   (b) najlepszymi parametrami znalezionymi przez GridSearchCV
# Wykres grupowany pokazuje zysk (lub brak) z tuningu.

print("Ewaluacja optymalnych modeli na zbiorze testowym...")

# Wyniki PRZED tuningiem (już w all_results z sekcji 2)
before = {name: r['f1_macro'] for name, r in all_results.items()}

# Wyniki PO tuningu
after = {}
for name, res in tuning_results.items():
    y_pred_tuned = res['best_model'].predict(X_test)
    f1 = f1_score(y_test, y_pred_tuned, average='macro', zero_division=0)
    after[name] = f1
    print(f"  {name:25s}: przed={before[name]:.4f}  po={f1:.4f}  Δ={f1-before[name]:+.4f}")

# ── Wykres słupkowy grupowany ────────────────────────────────────────────
model_names = list(before.keys())
x = np.arange(len(model_names))
width = 0.38

fig, ax = plt.subplots(figsize=(13, 6))
bars_before = ax.bar(x - width/2, [before[m] for m in model_names],
                     width, label='Domyślne parametry', color='steelblue', alpha=0.8)
bars_after  = ax.bar(x + width/2, [after[m]  for m in model_names],
                     width, label='Optymalne parametry (GridSearchCV)', color='coral', alpha=0.8)

# Adnotacje nad słupkami
for bar in bars_before:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9, color='steelblue')
for bar in bars_after:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9, color='coral')

# Strzałki pokazujące zmianę (delta)
for i, name in enumerate(model_names):
    delta = after[name] - before[name]
    mid_x = x[i] + width/2 + 0.02
    y_mid = max(before[name], after[name]) + 0.035
    ax.annotate(
        f'Δ={delta:+.3f}',
        xy=(x[i], y_mid), fontsize=8.5,
        ha='center', color='green' if delta >= 0 else 'red', fontweight='bold'
    )

ax.set_xticks(x)
ax.set_xticklabels(model_names, rotation=20, ha='right', fontsize=11)
ax.set_ylabel('F1-macro (zbiór testowy)', fontsize=12)
ax.set_title('Wpływ dostrajania hiperparametrów na F1-macro\n(domyślne parametry vs GridSearchCV)', fontweight='bold')
ax.legend(fontsize=11)
ax.set_ylim(0, min(1.0, max(list(after.values()) + list(before.values())) + 0.15))
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../results/tuning_before_after_comparison.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# PODSUMOWANIE: Tabela najlepszych parametrów
# ═══════════════════════════════════════════════════════════════════════════
print("\n{'='*60}")
print("NAJLEPSZE HIPERPARAMETRY (GridSearchCV, cv=3, F1-macro)")
print("{'='*60}")

rows = []
for name, res in tuning_results.items():
    y_pred_tuned = res['best_model'].predict(X_test)
    acc   = accuracy_score(y_test, y_pred_tuned)
    f1    = f1_score(y_test, y_pred_tuned, average='macro', zero_division=0)
    rows.append({
        'Model':          name,
        'Najlepsze parametry': str(res['best_params']),
        'CV F1-macro':    round(res['best_score'], 4) if res['best_score'] else '-',
        'Test Accuracy':  round(acc, 4),
        'Test F1-macro':  round(f1, 4),
    })

tuning_summary = pd.DataFrame(rows).sort_values('Test F1-macro', ascending=False)
display(tuning_summary)

# Zapis wyoptymalizowanego najlepszego modelu
import pickle
best_tuned_name = tuning_summary.iloc[0]['Model']
best_tuned_model = tuning_results[best_tuned_name]['best_model']

with open('../models/best_tuned_classical_model.pkl', 'wb') as f:
    pickle.dump({
        'model':       best_tuned_model,
        'model_name':  best_tuned_name,
        'best_params': tuning_results[best_tuned_name]['best_params'],
        'label_encoder': le,
    }, f)

print(f"\nNajlepszy wyoptymalizowany model: {best_tuned_name}")
print(f"Parametry: {tuning_results[best_tuned_name]['best_params']}")
print(f"Zapisano do: models/best_tuned_classical_model.pkl")

---
## 8. Analiza wpływu augmentacji danych na jakość klasyfikacji

Augmentacja danych polega na tworzeniu zmodyfikowanych kopii próbek treningowych,
co sztucznie powiększa zbiór i uczy model odporności na naturalne warianty sygnału.

Zaimplementowane techniki (z `src/augmentation.py`):

| Technika | Co symuluje |
|----------|-------------|
| **Szum gaussowski** | Szumy elektryczne elektrody/wzmacniacza |
| **Przesunięcie czasowe** | Różny punkt startowy nagrania |
| **Skalowanie amplitudy** | Różna czułość urządzenia (±20%) |
| **Dryft bazowy** | Ruch pacjenta, oddech (0.05–0.5 Hz) |
| **Maskowanie odcinka** | Chwilowa utrata kontaktu elektrody |

**Analiza:** Trenujemy najlepsze 2 modele (RF + SVM) na:
- Oryginalnych danych (bez augmentacji)  
- Danych z augmentacją (zbiór 2× większy)  

Porównujemy F1-macro na tym samym zbiorze testowym.

In [ ]:
import sys
sys.path.insert(0, '..')
from src.augmentation import (
    add_gaussian_noise, time_shift, scale_amplitude,
    add_baseline_wander, time_masking,
    AUGMENTATION_NAMES, augment_dataset,
)

# ═══════════════════════════════════════════════════════════════════════════
# WIZUALIZACJA 1: Przykład każdej techniki augmentacji na sygnale EKG
# ═══════════════════════════════════════════════════════════════════════════
# Bierzemy jeden przykładowy sygnał i pokazujemy jak wyglada po każdej
# transformacji – odprowadzenie II (indeks 1) jest najczytelniejsze.

rng_demo = np.random.default_rng(42)

# Weź pierwszy sygnał z X_train_proc (surowy przetworzony, przed ekstrakcją cech)
sample_ecg = X_train_proc[0]      # (n_samples, 12)
lead_idx   = 1                    # odprowadzenie II

augmented_samples = {
    'Oryginał':              sample_ecg,
    'Szum gaussowski':       add_gaussian_noise(sample_ecg, snr_db=20.0, rng=np.random.default_rng(0)),
    'Przesunięcie czasowe':  time_shift(sample_ecg, 0.1, rng=np.random.default_rng(1)),
    'Skalowanie amplitudy':  scale_amplitude(sample_ecg, (0.8, 1.2), rng=np.random.default_rng(2)),
    'Dryft bazowy':          add_baseline_wander(sample_ecg, fs=FS, amplitude=0.15, rng=np.random.default_rng(3)),
    'Maskowanie odcinka':    time_masking(sample_ecg, 0.05, rng=np.random.default_rng(4)),
}

fig, axes = plt.subplots(6, 1, figsize=(14, 14), sharex=True)
colors = ['black', 'steelblue', 'darkorange', 'green', 'red', 'purple']

for ax, (name, sig), color in zip(axes, augmented_samples.items(), colors):
    t = np.arange(sig.shape[0]) / FS   # oś czasu w sekundach
    ax.plot(t, sig[:, lead_idx], color=color, linewidth=0.9)
    ax.set_ylabel(name, fontsize=9, rotation=0, ha='right', labelpad=5)
    ax.grid(True, alpha=0.3)
    # Dla oryginału dodaj opis
    if name == 'Oryginał':
        ax.set_title('Techniki augmentacji danych EKG (odprowadzenie II)', fontweight='bold', fontsize=13)

axes[-1].set_xlabel('Czas [s]', fontsize=11)
plt.tight_layout()
plt.savefig('../results/augmentation_techniques_demo.png', bbox_inches='tight', dpi=150)
plt.show()
print("Wizualizacja technik augmentacji – odprowadzenie II")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# WIZUALIZACJA 2: Wpływ augmentacji na rozkład cech (histogram)
# ═══════════════════════════════════════════════════════════════════════════
# Pokazujemy jak augmentacja zmienia rozkład wybranej cechy (std odprow. II).
# Dobra augmentacja powinna poszerzać rozkład (większa różnorodność),
# ale nie deformować go drastycznie.

# Indeks cechy: std odprowadzenia II (cecha 13 w wektorze 228-wymiarowym)
# (19 cech na odprowadzenie × 1 (odprowadzenie II) + 1 (indeks std) = 1*19 + 1 = 20)
feat_idx  = 20    # std lead II
feat_name = 'Std – odprowadzenie II'

from src.feature_extraction import extract_features_batch

# Augmentuj dane treningowe
print("Augmentacja danych treningowych...")
X_train_aug_raw, y_train_aug = augment_dataset(
    X_train_proc, y_train,
    techniques=['noise', 'shift', 'scale', 'wander', 'mask'],
    augment_factor=1,   # podwojenie: 1 dodatkowa kopia na próbkę
    fs=FS, seed=42, verbose=True,
)

# Ekstrakcja cech z augmentowanego zbioru
print("\nEkstrakcja cech z augmentowanego zbioru...")
X_train_aug_feat = extract_features_batch(X_train_aug_raw, fs=FS, verbose=False)
X_train_aug_feat = np.nan_to_num(X_train_aug_feat, nan=0.0, posinf=0.0, neginf=0.0)

print(f"\nRozmiar oryginalny:    {X_train.shape}")
print(f"Rozmiar po augmentacji: {X_train_aug_feat.shape}")

# Histogram cechy przed/po augmentacji
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(X_train[:, feat_idx], bins=30, alpha=0.8, color='steelblue',
             label=f'Oryginalne (n={len(X_train)})', density=True)
axes[0].set_title(f'Bez augmentacji\n({feat_name})', fontweight='bold')
axes[0].set_xlabel('Wartość cechy')
axes[0].set_ylabel('Gęstość')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].hist(X_train[:, feat_idx], bins=30, alpha=0.6, color='steelblue',
             label=f'Oryginalne (n={len(X_train)})', density=True)
axes[1].hist(X_train_aug_feat[:, feat_idx], bins=30, alpha=0.5, color='coral',
             label=f'Po augmentacji (n={len(X_train_aug_feat)})', density=True)
axes[1].set_title(f'Z augmentacją – nakładka\n({feat_name})', fontweight='bold')
axes[1].set_xlabel('Wartość cechy')
axes[1].set_ylabel('Gęstość')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Wpływ augmentacji na rozkład cech statystycznych', fontweight='bold', fontsize=12)
plt.tight_layout()
plt.savefig('../results/augmentation_feature_distribution.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# WIZUALIZACJA 3: Porównanie każdej techniki z osobna – F1-macro (RF)
# ═══════════════════════════════════════════════════════════════════════════
# Testujemy każdą technikę augmentacji z osobna na modelu Random Forest,
# żeby zrozumieć która przynosi największy zysk.

from sklearn.ensemble import RandomForestClassifier

TECH_KEYS  = ['noise', 'shift', 'scale', 'wander', 'mask']
TECH_NAMES = ['Szum\ngaussowski', 'Przesunięcie\nczasowe',
              'Skalowanie\namplitudy', 'Dryft\nbazowy', 'Maskowanie\nodcinka']

rf_baseline_f1 = all_results['Random Forest']['f1_macro']
tech_f1 = {}

print("Trening RF dla każdej techniki augmentacji...")
for tech_key, tech_name in zip(TECH_KEYS, TECH_NAMES):
    # Augmentacja tylko jedną techniką
    X_aug_tech, y_aug_tech = augment_dataset(
        X_train_proc, y_train,
        techniques=[tech_key], augment_factor=1,
        fs=FS, seed=42, verbose=False,
    )
    X_feat_tech = extract_features_batch(X_aug_tech, fs=FS, verbose=False)
    X_feat_tech = np.nan_to_num(X_feat_tech)

    rf = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)
    rf.fit(X_feat_tech, y_aug_tech)
    y_pred_tech = rf.predict(X_test)
    f1 = f1_score(y_test, y_pred_tech, average='macro', zero_division=0)
    tech_f1[tech_key] = f1
    print(f"  {tech_name.replace(chr(10), ' '):25s}: F1={f1:.4f}  Δ={f1-rf_baseline_f1:+.4f}")

# Wykres słupkowy: baseline + każda technika
fig, ax = plt.subplots(figsize=(11, 6))
x_pos = np.arange(len(TECH_KEYS) + 1)  # +1 dla baseline
labels = ['Bez\naugmentacji'] + TECH_NAMES
values = [rf_baseline_f1] + [tech_f1[k] for k in TECH_KEYS]
bar_colors = ['steelblue'] + ['coral' if v >= rf_baseline_f1 else 'salmon' for v in values[1:]]

bars = ax.bar(x_pos, values, color=bar_colors, alpha=0.85, width=0.6)

# Linia bazowa
ax.axhline(rf_baseline_f1, color='steelblue', linestyle='--', linewidth=1.5,
           alpha=0.7, label=f'Baseline (brak augmentacji) = {rf_baseline_f1:.3f}')

# Adnotacje
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

# Delta nad słupkami (z wyjątkiem baseline)
for i, (key, name) in enumerate(zip(TECH_KEYS, TECH_NAMES), start=1):
    delta = tech_f1[key] - rf_baseline_f1
    ax.text(i, values[i] + 0.018, f'Δ={delta:+.3f}',
            ha='center', fontsize=9, color='green' if delta >= 0 else 'red', fontweight='bold')

ax.set_xticks(x_pos)
ax.set_xticklabels(labels, fontsize=10)
ax.set_ylabel('F1-macro (zbiór testowy)', fontsize=12)
ax.set_title('Wpływ każdej techniki augmentacji na F1-macro (Random Forest)',
             fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylim(0, min(1.0, max(values) + 0.12))
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../results/augmentation_per_technique_rf.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# WIZUALIZACJA 4: Porównanie wszystkich technik łącznie + wpływ augment_factor
# ═══════════════════════════════════════════════════════════════════════════
# Sprawdzamy jak wielkość augmentacji (1x, 2x, 3x więcej danych)
# wpływa na F1-macro dla RF i SVM.

from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

models_to_test = {
    'Random Forest': RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42),
    'SVM (RBF)':     Pipeline([('scaler', StandardScaler()),
                                ('clf', SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42))]),
}

augment_factors = [0, 1, 2, 3]   # 0 = brak augmentacji
results_aug = {name: [] for name in models_to_test}

print("Trening modeli przy różnych współczynnikach augmentacji...")
for factor in augment_factors:
    if factor == 0:
        # Brak augmentacji – użyj cech z sekcji 2
        X_tr = X_train
        y_tr = y_train
    else:
        X_aug_f, y_aug_f = augment_dataset(
            X_train_proc, y_train,
            techniques=['noise', 'shift', 'scale', 'wander', 'mask'],
            augment_factor=factor, fs=FS, seed=42, verbose=False,
        )
        X_tr = extract_features_batch(X_aug_f, fs=FS, verbose=False)
        X_tr = np.nan_to_num(X_tr)
        y_tr = y_aug_f

    print(f"\n  augment_factor={factor} → rozmiar zbioru treningowego: {len(X_tr)}")

    for model_name, model in models_to_test.items():
        import copy
        m = copy.deepcopy(model)
        m.fit(X_tr, y_tr)
        y_p = m.predict(X_test)
        f1  = f1_score(y_test, y_p, average='macro', zero_division=0)
        results_aug[model_name].append(f1)
        print(f"    {model_name:20s}: F1={f1:.4f}")

# ── Wykres liniowy ────────────────────────────────────────────────────────
dataset_sizes = [len(X_train) * (1 + f) for f in augment_factors]
fig, ax = plt.subplots(figsize=(10, 6))

colors_aug = ['steelblue', 'coral']
for (model_name, f1_list), color in zip(results_aug.items(), colors_aug):
    ax.plot(augment_factors, f1_list, 'o-', color=color,
            linewidth=2.5, markersize=9, label=model_name)
    # Annotacje punktów
    for af, f1 in zip(augment_factors, f1_list):
        ax.annotate(f'{f1:.3f}', (af, f1), textcoords='offset points',
                    xytext=(0, 10), ha='center', fontsize=9, color=color)

ax.set_xticks(augment_factors)
ax.set_xticklabels([
    f'Brak\n(n={dataset_sizes[0]})',
    f'×1\n(n={dataset_sizes[1]})',
    f'×2\n(n={dataset_sizes[2]})',
    f'×3\n(n={dataset_sizes[3]})',
])
ax.set_xlabel('Współczynnik augmentacji (wszystkie 5 technik)', fontsize=12)
ax.set_ylabel('F1-macro (zbiór testowy)', fontsize=12)
ax.set_title('Wpływ rozmiaru augmentacji na jakość klasyfikacji\n(wszystkie techniki łącznie)',
             fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/augmentation_factor_comparison.png', bbox_inches='tight', dpi=150)
plt.show()

print("\nPodsumowanie wyników augmentacji:")
print(f"{'Augment factor':<18} {'RF F1':>10} {'SVM F1':>10}")
for i, af in enumerate(augment_factors):
    rf_f1  = results_aug['Random Forest'][i]
    svm_f1 = results_aug['SVM (RBF)'][i]
    print(f"  {af:<16} {rf_f1:>10.4f} {svm_f1:>10.4f}")